# Legendre truncation

#### Import general modules

mpi4py is always required when using these tools. Numpy is always good to have if any manipulation is to be done.

In [52]:
# Import required modules
from mpi4py import MPI #equivalent to the use of MPI_init() in C
import matplotlib.pyplot as plt
import numpy as np

# Get mpi info
comm = MPI.COMM_WORLD

import os
os.environ["PYSEMTOOLS_DEBUG"] = 'false'
os.environ["PYSEMTOOLS_HIDE_LOG"] = 'false'

def get_folder_size(folder):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(folder):
        for filename in filenames:
            file_path = os.path.join(dirpath, filename)
            total_size += os.path.getsize(file_path)
    return total_size

In [53]:
# Download the data if it does not exist
sem_data_path = "../../data/sem_data/"
if not os.path.exists(sem_data_path):
    print("Sem data not found, cloning repository...")
    os.system(f"git clone https://github.com/adperezm/sem_data.git {sem_data_path}")
else:
    print("Sem data found.")

Sem data found.


#### Import modules from pysemtools

In this case we will import all the data types that we currently support, as well as io functions that are required to populate them.

In [54]:
# Data types
from pysemtools.datatypes.msh import Mesh
from pysemtools.datatypes.coef import Coef
from pysemtools.datatypes.field import Field, FieldRegistry

# Readers
from pysemtools.io.ppymech.neksuite import preadnek, pynekread

# Writers
from pysemtools.io.ppymech.neksuite import pwritenek, pynekwrite

fname = '../../data/sem_data/instantaneous/channel_nelv_512/field0.f01601'

## Read the data

In [55]:
# Read the data
msh = Mesh(comm, create_connectivity=False)
fld = FieldRegistry(comm)
pynekread(fname, comm, data_dtype=np.single, msh = msh, fld = fld)
pynekwrite("test_no_mesh", comm, msh = msh, fld = fld, write_mesh=False)
 
# Get the coefficients
coef = Coef(msh=msh, comm=comm)

2026-08-06 17:01:00,922  pynekread             INFO      Reading file: ../../data/sem_data/instantaneous/channel_nelv_512/field0.f01601
2026-08-06 17:01:00,943  Mesh                  INFO      Mesh object initialized from coordinates with type: float32 - Elapsed time: 0.016519748000064283s
2026-08-06 17:01:00,947  Field                 INFO      Field registry updated with: ['u', 'v', 'w', 'p'] - dtype: float32
2026-08-06 17:01:00,948  pynekread             INFO      File successfully read - Elapsed time: 0.025733048999882158s
2026-08-06 17:01:00,948  pynekwrite            INFO      Writing file: test_no_mesh
2026-08-06 17:01:00,953  pynekwrite            INFO      File written - Elapsed time: 0.004370950000065932s
2026-08-06 17:01:00,981  Coef                  INFO      Coef object initialized - dtype: float32 - Elapsed time: 0.027327446000072086s


## Compress the data with GPR

### Initialize the Direct sampler

In [56]:
from pysemtools.compression.legendre_truncation import DiscreetLegendreTruncation
from pysemtools.compression.legendre_truncation_bitplane import DiscreetLegendreTruncationBP
from pysemtools.compression.legendre_truncation_bitplaneV2 import DiscreetLegendreTruncationBPAdaptive
from pysemtools.compression.zfp_wrapper import ZFPWrapper

# Initialize all samplers with numpy backend
dlt = DiscreetLegendreTruncation(comm=comm, msh=msh, coef=coef)
dlt_bp = DiscreetLegendreTruncationBP(comm=comm, msh=msh, coef=coef)
dlt_bp_v2 = DiscreetLegendreTruncationBPAdaptive(comm=comm, msh=msh, coef=coef)
dlt_zfp = ZFPWrapper(comm=comm, msh=msh)

2026-08-06 17:01:00,989  DirectSampler         INFO      Initializing the DirectSampler from a Mesh object
2026-08-06 17:01:00,997  DirectSampler         INFO      Initializing the DirectSampler from a Mesh object
2026-08-06 17:01:01,004  DirectSampler         INFO      Initializing the DirectSampler from a Mesh object
2026-08-06 17:01:01,012  DirectSampler         INFO      Initializing the DirectSampler from a Mesh object


### Sample the data

In [57]:
# Select the options
target_error = 1e-2
zfp_bitrate_percent_levels = [i / 100 for i in range(1, 100)]
field_names = list(fld.registry.keys())

for fld_name in field_names:

    # Existing fixed-error type
    dlt.log.tic()
    dlt.sample_field(field=fld.registry[fld_name], field_name=fld_name, compression_method="fixed_error", target_error=target_error)
    dlt.log.toc()

    # Bitplane V1
    dlt_bp.log.tic()
    dlt_bp.sample_field(field=fld.registry[fld_name], field_name=fld_name, target_error=target_error)
    dlt_bp.log.toc()

    # Bitplane V2 (adaptive)
    dlt_bp_v2.log.tic()
    dlt_bp_v2.sample_field(field=fld.registry[fld_name], field_name=fld_name, target_error=target_error)
    dlt_bp_v2.log.toc()

    # ZFP is handled in a dedicated bitrate sweep cell below (compress+decompress+selection).

2026-08-06 17:01:01,092  DirectSampler         INFO      Sampling the field with options: covariance_method: {covariance_method}, compression_method: {compression_method}
2026-08-06 17:01:01,097  DirectSampler         INFO      Transforming the field into to legendre space


2026-08-06 17:01:01,242  DirectSampler         INFO      Sampling the field using the fixed error method. using settings: {'method': 'fixed_error', 'target_error': 0.01, 'max_samples_per_it': 1}
2026-08-06 17:01:01,256  DirectSampler         INFO      Sampled_field saved in field uncompressed_data["u"]["field"]
2026-08-06 17:01:01,257  DirectSampler         INFO      Elapsed time: 0.16477273300006345s
2026-08-06 17:01:01,258  DirectSampler         INFO      Sampling field "u" with target_error=0.01
2026-08-06 17:01:01,258  DirectSampler         INFO      Transforming the field into to legendre space
2026-08-06 17:01:01,359  DirectSampler         INFO      Sampling the field using bitplane coding. using settings: {'method': 'fixed_error_bitplane', 'target_error': 0.01}
2026-08-06 17:01:02,382  DirectSampler         INFO      Bitplane stream saved for field "u"
2026-08-06 17:01:02,382  DirectSampler         INFO      Elapsed time: 1.1249075529999573s
2026-08-06 17:01:02,383  DirectSample

### Encode it

In [58]:
dlt.compress_samples(lossless_compressor="bzip2")
dlt_bp.compress_samples(lossless_compressor="bzip2")
dlt_bp_v2.compress_samples(lossless_compressor="bzip2")

2026-08-06 17:01:08,971  DirectSampler         INFO      Compressing the data using the lossless compressor: bzip2
2026-08-06 17:01:08,972  DirectSampler         INFO      Compressing data in uncompressed_data
2026-08-06 17:01:08,973  DirectSampler         INFO      Compressing data for field ["u"]:
2026-08-06 17:01:08,973  DirectSampler         INFO      Compressing ["field"] for field ["u"]
2026-08-06 17:01:09,015  DirectSampler         INFO      Compressing data for field ["v"]:
2026-08-06 17:01:09,016  DirectSampler         INFO      Compressing ["field"] for field ["v"]
2026-08-06 17:01:09,042  DirectSampler         INFO      Compressing data for field ["w"]:
2026-08-06 17:01:09,043  DirectSampler         INFO      Compressing ["field"] for field ["w"]
2026-08-06 17:01:09,076  DirectSampler         INFO      Compressing data for field ["p"]:
2026-08-06 17:01:09,077  DirectSampler         INFO      Compressing ["field"] for field ["p"]
2026-08-06 17:01:09,097  DirectSampler        

### Write it out

In [59]:
# ZFP fixed-bitrate sweep on the first field only: select the first bitrate that meets target_error
# using the same physical-space B-weighted mean RMS metric used in this notebook.
zfp_selected = {}
dlt_zfp = ZFPWrapper(comm=comm, msh=msh)
B = coef.B
den_B = np.sum(B, axis=(1, 2, 3))

first_fld_name = field_names[0]
u_first = fld.registry[first_fld_name]

# Use the actual source dtype for bitrate scaling (e.g., 32 for float32 fields).
dtype_bits = u_first.dtype.itemsize * 8
print(f"[{first_fld_name}] dtype={u_first.dtype}, dtype_bits={dtype_bits}")

zfp_sweep_results = []
zfp_selected_bitrate = None
zfp_selected_error = None
zfp_selected_percent = None

for pct in zfp_bitrate_percent_levels:
    bitrate = pct * dtype_bits

    zfp_trial = ZFPWrapper(comm=comm, msh=msh)
    zfp_trial.sample_field(
        field=u_first,
        field_name=first_fld_name,
        compression_method="fixed_bitrate",
        bitrate=bitrate,
    )
    zfp_trial.compress_samples()

    trial_data = zfp_trial.decompress_samples(zfp_trial.settings, zfp_trial.compressed_data)
    if first_fld_name not in trial_data:
        raise KeyError(f"ZFP trial reconstruction missing field '{first_fld_name}'.")
    u_trial = trial_data[first_fld_name]["field"]

    err_trial = u_trial - u_first
    num_B_trial = np.sum((err_trial**2) * B, axis=(1, 2, 3))
    mean_weighted_rms_trial = np.mean(np.sqrt(num_B_trial / den_B))
    zfp_sweep_results.append((pct, bitrate, mean_weighted_rms_trial))

    print(
        f"[{first_fld_name}] zfp level={pct:.0%}, rate={bitrate:.2f} bits/value, "
        f"mean B-weighted RMS={mean_weighted_rms_trial:.3e}"
    )

    if mean_weighted_rms_trial <= target_error:
        zfp_selected_bitrate = bitrate
        zfp_selected_error = mean_weighted_rms_trial
        zfp_selected_percent = pct
        break

if zfp_selected_bitrate is None:
    raise RuntimeError(
        f"No ZFP percentage level met target_error for field '{first_fld_name}'. "
        "Expand zfp_bitrate_percent_levels or relax target_error."
    )

# Keep the selected ZFP bitrate global and reuse it for every field.
zfp_selected = {
    "reference_field": first_fld_name,
    "bitrate": zfp_selected_bitrate,
    "percent": zfp_selected_percent,
    "error": zfp_selected_error,
    "dtype_bits": dtype_bits,
    "sweep": zfp_sweep_results,
}

# Apply the selected ZFP bitrate to all fields and add them to the final multi-field ZFP object.
for fld_name in field_names:
    u = fld.registry[fld_name]

    dlt_zfp.sample_field(
        field=u,
        field_name=fld_name,
        compression_method="fixed_bitrate",
        bitrate=zfp_selected["bitrate"],
    )

    print(
        f"[{fld_name}] Reusing ZFP level={zfp_selected['percent']:.0%}, "
        f"rate={zfp_selected['bitrate']:.2f} bits/value, "
        f"mean B-weighted RMS={zfp_selected['error']:.3e}"
    )

# Write only the selected ZFP case to disk for fair file-size comparison.
dlt_zfp.compress_samples()

# Avoid stale output artifacts from previous runs when comparing folder sizes.
for out_name in ["test", "test_bp", "test_bp_v2", "test_zfp"]:
    comp_folder = f"{out_name}_comp"
    if os.path.isdir(comp_folder):
        for fname_local in os.listdir(comp_folder):
            os.remove(os.path.join(comp_folder, fname_local))
    if os.path.isfile(out_name):
        os.remove(out_name)

dlt.write_compressed_samples(comm=comm, filename="test")
dlt_bp.write_compressed_samples(comm=comm, filename="test_bp")
dlt_bp_v2.write_compressed_samples(comm=comm, filename="test_bp_v2")
dlt_zfp.write_compressed_samples(comm=comm, filename="test_zfp")

2026-08-06 17:01:09,289  DirectSampler         INFO      Initializing the DirectSampler from a Mesh object
[u] dtype=float32, dtype_bits=32
2026-08-06 17:01:09,317  DirectSampler         INFO      Initializing the DirectSampler from a Mesh object
2026-08-06 17:01:09,346  DirectSampler         INFO      Compressing data in uncompressed_data
2026-08-06 17:01:09,347  DirectSampler         INFO      Compressing data for field ["u"]:
2026-08-06 17:01:09,348  DirectSampler         INFO      Compressing ["field"] for field ["u"]
[u] zfp level=1%, rate=0.32 bits/value, mean B-weighted RMS=4.177e-02
2026-08-06 17:01:09,356  DirectSampler         INFO      Initializing the DirectSampler from a Mesh object
2026-08-06 17:01:09,466  DirectSampler         INFO      Compressing data in uncompressed_data
2026-08-06 17:01:09,467  DirectSampler         INFO      Compressing data for field ["u"]:
2026-08-06 17:01:09,468  DirectSampler         INFO      Compressing ["field"] for field ["u"]
[u] zfp level=

## Decompress the data

### Read the data

In [60]:
dlt_read = DiscreetLegendreTruncation(comm, filename="test")
dlt_bp_read = DiscreetLegendreTruncationBP(comm, filename="test_bp")
dlt_bp_v2_read = DiscreetLegendreTruncationBPAdaptive(comm, filename="test_bp_v2")
dlt_zfp_read = ZFPWrapper(comm, filename="test_zfp")

2026-08-06 17:01:10,110  DirectSampler         INFO      Initializing the DirectSampler from file: test
2026-08-06 17:01:10,181  DirectSampler         INFO      Initializing the DirectSampler from file: test_bp
2026-08-06 17:01:10,214  DirectSampler         INFO      Initializing the DirectSampler from file: test_bp_v2
2026-08-06 17:01:10,244  DirectSampler         INFO      Initializing the DirectSampler from file: test_zfp


### Reconstruct the data from the samples

In [61]:
# Recreate fresh readers from files to ensure reconstruction is strictly file-backed
dlt_read = DiscreetLegendreTruncation(comm, filename="test")
dlt_bp_read = DiscreetLegendreTruncationBP(comm, filename="test_bp")
dlt_bp_v2_read = DiscreetLegendreTruncationBPAdaptive(comm, filename="test_bp_v2")
dlt_zfp_read = ZFPWrapper(comm, filename="test_zfp")

u_np = {}
u_bp = {}
u_bp_v2 = {}
u_zfp = {}

for fld_name in field_names:
    u_np[fld_name] = dlt_read.reconstruct_field(field_name=fld_name)
    u_bp[fld_name] = dlt_bp_read.reconstruct_field(field_name=fld_name)
    u_bp_v2[fld_name] = dlt_bp_v2_read.reconstruct_field(field_name=fld_name)
    u_zfp[fld_name] = dlt_zfp_read.reconstruct_field(field_name=fld_name)

    if u_np[fld_name] is None or u_bp[fld_name] is None or u_bp_v2[fld_name] is None or u_zfp[fld_name] is None:
        raise RuntimeError(f"Reconstruction returned None for field '{fld_name}'.")

2026-08-06 17:01:10,304  DirectSampler         INFO      Initializing the DirectSampler from file: test
2026-08-06 17:01:10,410  DirectSampler         INFO      Initializing the DirectSampler from file: test_bp
2026-08-06 17:01:10,442  DirectSampler         INFO      Initializing the DirectSampler from file: test_bp_v2
2026-08-06 17:01:10,471  DirectSampler         INFO      Initializing the DirectSampler from file: test_zfp


## Additional checks: bitplane payload and metrics

## Visualize the data

In [62]:
# General settings
plot_fields = ["u", "v"] #list(field_names)
u_levels_by_field = {}
err_levels_by_field = {}

for fld_name in plot_fields:
    u = fld.registry[fld_name]
    u_levels_by_field[fld_name] = np.linspace(np.min(u), np.max(u), 100)

    err_fixed = np.abs(u_np[fld_name] - u)
    err_bp = np.abs(u_bp[fld_name] - u)
    err_bp_v2 = np.abs(u_bp_v2[fld_name] - u)
    err_zfp = np.abs(u_zfp[fld_name] - u)
    err_max = np.max([np.max(err_fixed), np.max(err_bp), np.max(err_bp_v2), np.max(err_zfp)])

    if err_max <= 0:
        err_max = 1e-15

    err_levels_by_field[fld_name] = 100

### Numpy

## Test the derivative

## Metrics

Check reconstruction error against the fixed-error target.

In [63]:
# B-weighted physical-space reconstruction error per field + tabular reporting.
def _format_bytes(nbytes):
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(nbytes)
    for unit in units:
        if value < 1024.0 or unit == units[-1]:
            return f"{value:.2f} {unit}"
        value /= 1024.0

def _size_bytes_using_existing(path):
    # Reuse the existing notebook helper when possible.
    if os.path.isfile(path):
        return os.path.getsize(path)
    return get_folder_size(path)

def _print_ascii_table(headers, rows):
    str_rows = [[str(v) for v in row] for row in rows]
    widths = [len(str(h)) for h in headers]
    for row in str_rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(value))

    def _fmt_row(values):
        return "| " + " | ".join(str(values[i]).ljust(widths[i]) for i in range(len(values))) + " |"

    sep = "+-" + "-+-".join("-" * w for w in widths) + "-+"
    print(sep)
    print(_fmt_row(headers))
    print(sep)
    for row in str_rows:
        print(_fmt_row(row))
    print(sep)

B = coef.B
den_B = np.sum(B, axis=(1, 2, 3))
metrics_by_field = {}

for fld_name in plot_fields:
    u = fld.registry[fld_name]

    err_fixed = u_np[fld_name] - u
    err_bp = u_bp[fld_name] - u
    err_bp_v2 = u_bp_v2[fld_name] - u
    err_zfp = u_zfp[fld_name] - u

    rms_fixed = np.mean(np.sqrt(np.sum((err_fixed**2) * B, axis=(1, 2, 3)) / den_B))
    rms_bp = np.mean(np.sqrt(np.sum((err_bp**2) * B, axis=(1, 2, 3)) / den_B))
    rms_bp_v2 = np.mean(np.sqrt(np.sum((err_bp_v2**2) * B, axis=(1, 2, 3)) / den_B))
    rms_zfp = np.mean(np.sqrt(np.sum((err_zfp**2) * B, axis=(1, 2, 3)) / den_B))

    metrics_by_field[fld_name] = {
        "fixed_error": rms_fixed,
        "bitplane": rms_bp,
        "bitplane_v2": rms_bp_v2,
        "zfp": rms_zfp,
    }

print("\n=== Reconstruction Error by Method (mean B-weighted RMS) ===")
for method in ["fixed_error", "bitplane", "bitplane_v2", "zfp"]:
    print(f"\n--- {method} ---")
    rows = [[fld_name, f"{metrics_by_field[fld_name][method]:.3e}"] for fld_name in plot_fields]
    _print_ascii_table(["field", "error"], rows)

# File-size comparison: base vs compressed datasets.
num_values = int(sum(fld.registry[fld_name].size for fld_name in plot_fields))
base_path = "test_no_mesh"
compressed_paths = {
    "fixed_error": "test_comp",
    "bitplane": "test_bp_comp",
    "bitplane_v2": "test_bp_v2_comp",
    "zfp": "test_zfp_comp",
}

base_size = _size_bytes_using_existing(base_path)

print("\n=== Size Summary ===")
print(f"base ({base_path}): {base_size} bytes ({_format_bytes(base_size)})")
print(f"total doubles in all fields: {num_values}")

size_rows = []
for method, comp_path in compressed_paths.items():
    comp_size = _size_bytes_using_existing(comp_path)
    ratio_base_over_comp = (base_size / comp_size) if comp_size > 0 else np.nan
    ratio_comp_over_base = (comp_size / base_size) if base_size > 0 else np.nan
    bits_per_double = comp_size / base_size * 32 if base_size > 0 else np.nan

    ratio_boc_txt = f"{ratio_base_over_comp:.3f}x" if np.isfinite(ratio_base_over_comp) else "n/a"
    ratio_cob_txt = f"{ratio_comp_over_base:.3f}x" if np.isfinite(ratio_comp_over_base) else "n/a"
    bpd_txt = f"{bits_per_double:.3f}" if np.isfinite(bits_per_double) else "n/a"

    size_rows.append([
        method,
        f"{comp_size}",
        _format_bytes(comp_size),
        ratio_boc_txt,
        ratio_cob_txt,
        bpd_txt,
    ])

_print_ascii_table(
    [
        "method",
        "compressed bytes",
        "compressed size",
        "ratio (base/comp)",
        "ratio (comp/base)",
        "bits/double",
    ],
    size_rows,
 )


=== Reconstruction Error by Method (mean B-weighted RMS) ===

--- fixed_error ---
+-------+-----------+
| field | error     |
+-------+-----------+
| u     | 9.850e-03 |
| v     | 9.782e-03 |
+-------+-----------+

--- bitplane ---
+-------+-----------+
| field | error     |
+-------+-----------+
| u     | 7.983e-03 |
| v     | 7.998e-03 |
+-------+-----------+

--- bitplane_v2 ---
+-------+-----------+
| field | error     |
+-------+-----------+
| u     | 9.889e-03 |
| v     | 9.845e-03 |
+-------+-----------+

--- zfp ---
+-------+-----------+
| field | error     |
+-------+-----------+
| u     | 9.050e-03 |
| v     | 5.426e-03 |
+-------+-----------+

=== Size Summary ===
base (test_no_mesh): 4212872 bytes (4.02 MB)
total doubles in all fields: 524288
+-------------+------------------+-----------------+-------------------+-------------------+-------------+
| method      | compressed bytes | compressed size | ratio (base/comp) | ratio (comp/base) | bits/double |
+-------------+-----